# Chapter 22 — Six Sigma & Quality Statistics
*Track 4: Engineers*

## 🎯 Learning Objectives
- Understand the Six Sigma quality standard and DPMO
- Implement measurement system analysis (Gage R&R)
- Apply hypothesis testing in a quality control context

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Six Sigma Quality

**Six Sigma** means the process mean is 6σ away from the nearest spec limit.
With a 1.5σ shift (Motorola convention):

| Sigma level | DPMO | Yield |
|-------------|------|-------|
| 3σ | 66,807 | 93.32% |
| 4σ | 6,210 | 99.38% |
| 5σ | 233 | 99.977% |
| 6σ | 3.4 | 99.9997% |

DPMO = Defects Per Million Opportunities

In [ ]:
# DPMO curve
sigma_levels = np.linspace(1, 6.5, 200)
shift = 1.5  # Motorola long-term shift
dpmo = (stats.norm.sf(sigma_levels - shift) + stats.norm.cdf(-(sigma_levels + shift))) * 1e6

plt.figure(figsize=(9, 5))
plt.semilogy(sigma_levels, dpmo)
for level in [3, 4, 5, 6]:
    d = (stats.norm.sf(level - 1.5) + stats.norm.cdf(-(level + 1.5))) * 1e6
    plt.scatter([level], [d], s=80, zorder=5)
    plt.annotate(f"{level}σ
{d:.0f} DPMO", (level, d), textcoords="offset points",
                 xytext=(10, 5), fontsize=9)
plt.xlabel("Sigma Level"); plt.ylabel("DPMO (log scale)")
plt.title("Sigma Level vs DPMO (with 1.5σ shift)")
plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Gage Repeatability & Reproducibility

The total measurement variation is:
$$\sigma^2_{total} = \sigma^2_{part} + \sigma^2_{gage}$$
$$\sigma^2_{gage} = \sigma^2_{repeatability} + \sigma^2_{reproducibility}$$

**%R&R**:
$$\%R\&R = \frac{\sigma_{gage}}{\sigma_{total}} \times 100$$

Rule: %R&R < 10% → acceptable; 10–30% → marginal; >30% → unacceptable.

In [ ]:
# Gage R&R study simulation
n_parts = 10
n_operators = 3
n_reps = 2

part_variation = 5.0    # true part-to-part std
repeatability  = 0.8    # within-operator std
reproducibility = 1.2   # between-operator std

part_means = rng.normal(100, part_variation, n_parts)
operator_biases = rng.normal(0, reproducibility, n_operators)

data = []
for p in range(n_parts):
    for o in range(n_operators):
        for r in range(n_reps):
            meas = (part_means[p] + operator_biases[o] +
                    rng.normal(0, repeatability))
            data.append({"part": p+1, "operator": o+1, "rep": r+1, "measurement": meas})
df = pd.DataFrame(data)

# ANOVA-based R&R
grand_mean = df["measurement"].mean()
# Between-part variance
part_means_est = df.groupby("part")["measurement"].mean()
ss_parts = n_operators * n_reps * ((part_means_est - grand_mean)**2).sum()
ms_parts = ss_parts / (n_parts - 1)
# Repeatability
ss_rep = df.groupby(["part","operator"])["measurement"].var(ddof=1).sum() * (n_reps - 1)
ms_rep = ss_rep / (n_parts * n_operators * (n_reps - 1))
sigma_repeat = np.sqrt(ms_rep)
sigma_total  = df["measurement"].std(ddof=1)
sigma_gage   = np.sqrt(max(0, ms_rep + (n_reps * max(0, ms_parts - ms_rep) / n_reps)))

perc_rr = sigma_repeat / sigma_total * 100
print(f"σ_repeatability: {sigma_repeat:.3f}")
print(f"σ_total:         {sigma_total:.3f}")
print(f"%R&R:            {perc_rr:.1f}%")
print("✅ Acceptable" if perc_rr < 10 else "⚠️ Marginal" if perc_rr < 30 else "❌ Unacceptable")

In [ ]:
# Hypothesis tests in quality context
# 1. Are two machine outputs the same mean? (Welch t-test)
machine_A = rng.normal(100.2, 1.5, 50)
machine_B = rng.normal(100.8, 1.7, 50)
t_stat, p_val = stats.ttest_ind(machine_A, machine_B, equal_var=False)
print(f"Machine comparison: t={t_stat:.3f}, p={p_val:.4f}")
print("Different" if p_val < 0.05 else "Same within noise")

# 2. Is variance in spec? F-test
spec_variance = 2.25  # σ = 1.5 → σ² = 2.25
n = len(machine_A)
chi2_stat = (n-1) * machine_A.var(ddof=1) / spec_variance
p_chi2 = 2 * min(stats.chi2.cdf(chi2_stat, n-1), stats.chi2.sf(chi2_stat, n-1))
print(f"
Variance test: χ²={chi2_stat:.3f}, p={p_chi2:.4f}")
print("Variance OK ✅" if p_chi2 > 0.05 else "Variance out of spec ❌")

## 🎯 Track 4 Complete! 🏆

**You've mastered:**
- ✅ Reliability & failure probability: Weibull, bathtub curve
- ✅ Poisson processes: inter-arrivals, system applications
- ✅ Queuing theory: M/M/1, Little's Law, capacity planning
- ✅ Statistical Process Control: X̄/R/p charts, Cp/Cpk
- ✅ Signal vs noise: SNR, filtering, Kalman filter
- ✅ Failure mode analysis: FMEA, fault trees, event trees
- ✅ Monte Carlo for engineering: uncertainty propagation, reliability
- ✅ Risk assessment: risk matrices, F-N curves, safety factors
- ✅ Regression: calibration, degradation, residual diagnostics
- ✅ Six Sigma: DPMO, Gage R&R, quality hypothesis tests

**Ready for Tier 3?** → [Chapter 23 — Markov Chains]